# EDA

## 1. Librerias

In [ ]:
import json, logging, gc
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import PurePosixPath
import numpy as np
from scipy.signal import butter, filtfilt
import boto3
import h5py
import numpy as np
import pandas as pd
import s3fs
import scipy.io as sio
from sklearn.preprocessing import MultiLabelBinarizer
from dx_mapping import map_dx, CATEGORIES

Entorno listo


## 2. Configuración global

In [ ]:
BASE_PATH = 'tfm-mu-bd/raw/'           
OUT_HDF5 = 's3://tfm-mu-bd/processed/deep_learning/dataset.h5'
OUT_PARQUET = 's3://tfm-mu-bd/processed/machine_learning/dataset.parquet'
OUT_META = 's3://tfm-mu-bd/processed/metadata/metadata.json'

LOCAL_HDF5 = './deep_learning.h5'
LOCAL_PARQUET = './ml_dataset.parquet'
LOCAL_META = './metadata.json'

EXPECTED_LEADS = 12
EXPECTED_LENGTH = 5000
MAX_WORKERS = 8    
CHUNK_SIZE = 500

s3 = s3fs.S3FileSystem()
print(f'Chunk size: {CHUNK_SIZE} registros  (~{CHUNK_SIZE*12*5000*4/1e6:.0f} MB/chunk)')

Chunk size: 500 registros  (~120 MB/chunk)


## 3. Listado de ficheros S3

In [59]:
def list_level2(s3, l1_folder):
    result = []
    try:
        for l2 in s3.ls(l1_folder):
            try:
                result.extend(s3.ls(l2))
            except Exception as e:
                log.warning('ls fallo en %s: %s', l2, e)
    except Exception as e:
        log.warning('ls fallo en %s: %s', l1_folder, e)
    return result


def list_all_files(s3, root):
    level1 = s3.ls(root)
    all_files = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(list_level2, s3, l1): l1 for l1 in level1}
        for fut in as_completed(futures):
            all_files.extend(fut.result())
    return ['s3://' + f for f in all_files]


all_files = list_all_files(s3, BASE_PATH)
print(f'Ficheros encontrados: {len(all_files)}')

Ficheros encontrados: 90755


## 4. Emparejamiento 

In [ ]:
def split_and_pair(files):
    mats, heas = {}, {}
    for f in files:
        stem = PurePosixPath(f).stem
        if f.endswith('.mat'):   mats[stem] = f
        elif f.endswith('.hea'): heas[stem] = f

    pairs = {rid: {'mat': mats[rid], 'hea': heas[rid]}
             for rid in mats if rid in heas}
    log.info('Pares válidos: %d | .mat sin .hea: %d | .hea sin .mat: %d',
             len(pairs), len(mats)-len(pairs), len(heas)-len(pairs))
    return pairs


pairs = split_and_pair(all_files)
pair_list = list(pairs.items())   
print(f'Pares (mat+hea): {len(pair_list)}')

INFO | Pares válidos: 45152 | .mat sin .hea: 0 | .hea sin .mat: 0


Pares (mat+hea): 45152


## 5. Funciones de parseo

In [ ]:
def parse_hea(lines):
    age, sex, dx = None, None, []
    for line in lines:
        line = line.strip()
        if line.startswith('#Age'):
            try:    age = float(line.split(':', 1)[1].strip())
            except: age = np.nan
        elif line.startswith('#Sex'):
            val = line.split(':', 1)[1].strip().lower()
            sex = {'male': 1, 'female': 0}.get(val, -1)
        elif line.startswith('#Dx'):
            try:    dx = [x.strip() for x in line.split(':', 1)[1].split(',') if x.strip()]
            except: dx = []
    return age, sex, dx


def extract_metadata(hea_path):
    with s3.open(hea_path, 'r') as f:
        lines = f.read().splitlines()
    age, sex, dx = parse_hea(lines)
    return {'age': age, 'sex': sex, 'dx': dx}


def load_ecg(mat_path):
    try:
        with s3.open(mat_path, 'rb') as f:
            mat = sio.loadmat(f)
    except Exception as e:
        log.debug('Error cargando %s: %s', mat_path, e)
        return None

    signal = next(
        (v for k, v in mat.items() if not k.startswith('__') and isinstance(v, np.ndarray)),
        None
    )
    if signal is None or signal.ndim != 2 or EXPECTED_LEADS not in signal.shape:
        return None
    if signal.shape[1] == EXPECTED_LEADS:
        signal = signal.T
    return signal.astype(np.float32)


def build_record(rid, paths):
    try:
        meta = extract_metadata(paths['hea'])
        signal = load_ecg(paths['mat'])
        if signal is None or signal.shape != (EXPECTED_LEADS, EXPECTED_LENGTH):
            return None
        if not meta['dx']:
            return None
        return {
            'record_id': rid,
            'signal': signal,
            'age': float(meta['age']) if meta['age'] is not None else np.nan,
            'sex': int(meta['sex'])   if meta['sex'] is not None else -1,
            'dx': meta['dx']
        }
    except Exception as e:
        log.debug('Error en record %s: %s', rid, e)
        return None

## 6. Función de features

In [ ]:
# Filtro Butterworth

def butter_bandpass_filter(signal, lowcut=0.5, highcut=40, fs=500, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')

    filtered = np.zeros_like(signal, dtype=np.float32)

    for i in range(signal.shape[0]):
        filtered[i] = filtfilt(b, a, signal[i])

    return filtered

# Normalización Z-score

def zscore_normalize(signal):
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True) + 1e-8
    return (signal - mean) / std

def load_and_preprocess_record(s3, rid, paths, load_ecg_fn, extract_metadata_fn):

    try:
        meta = extract_metadata_fn(s3, paths["hea"])
        signal = load_ecg_fn(s3, paths["mat"])

        if signal is None:
            return None

        if signal.shape != (12, 5000):
            return None

        signal = butter_bandpass_filter(signal)
        
        signal = zscore_normalize(signal)

        return {
            "record_id": rid,
            "signal": signal.astype(np.float32),
            "age": meta.get("age"),
            "sex": meta.get("sex"),
            "dx": meta.get("dx")
        }

    except Exception as e:
        print(f"[ERROR] {rid}: {str(e)}")
        return None

def build_filtered_dataset(s3, pairs, load_ecg_fn, extract_metadata_fn, limit=None, log_every=1000):

    dataset = []
    failed = 0

    for i, (rid, paths) in enumerate(pairs.items()):

        if limit and i >= limit:
            break

        record = load_and_preprocess_record(
            s3,
            rid,
            paths,
            load_ecg_fn,
            extract_metadata_fn
        )

        if record is None:
            failed += 1
            continue

        dataset.append(record)

        if i % log_every == 0:
            print(f"[INFO] Procesados: {i} | válidos: {len(dataset)} | fallidos: {failed}")

    print("\n RESUMEN ")
    print("Registros válidos:", len(dataset))
    print("Fallidos:", failed)

    return dataset


FEATURE_NAMES = [
    f'lead{l}_{stat}'
    for stat in ['mean', 'std', 'min', 'max', 'energy']
    for l in range(EXPECTED_LEADS)
]

def extract_features_chunk(signals):

    return np.concatenate([
        signals.mean(axis=2),
        signals.std(axis=2),
        signals.min(axis=2),
        signals.max(axis=2),
        (signals ** 2).sum(axis=2)
    ], axis=1)

## 7. Pipeline principal por chunks


In [ ]:
dx_counter = Counter()
null_stats = {'missing_age': 0, 'missing_sex': 0, 'missing_dx': 0}
all_lengths = []
total_valid = 0
total_failed = 0

hdf5_file = h5py.File(LOCAL_HDF5, 'w')
hdf5_signal = None   # se inicializa en el primer chunk (necesitamos saber N total)
parquet_chunks = []   # acumulamos DataFrames pequeños y escribimos al final
write_idx = 0

for chunk_start in range(0, len(pair_list), CHUNK_SIZE):
    chunk = pair_list[chunk_start : chunk_start + CHUNK_SIZE]
    chunk_records = []

    # Descarga paralela del chunk
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(build_record, rid, paths): rid for rid, paths in chunk}
        for fut in as_completed(futures):
            r = fut.result()
            if r is None:
                total_failed += 1
            else:
                chunk_records.append(r)

    if not chunk_records:
        log.warning('Chunk %d vacío, saltando', chunk_start // CHUNK_SIZE)
        continue

    # EDA acumulada 
    for r in chunk_records:
        all_lengths.append(r['signal'].shape[1])
        if np.isnan(r['age']): null_stats['missing_age'] += 1
        if r['sex'] not in (0, 1): null_stats['missing_sex'] += 1
        if not r['dx']: null_stats['missing_dx']  += 1
        else: dx_counter.update(r['dx'])

    #  Stack del chunk (N_chunk, 12, 5000) 
    signals_chunk = np.stack([r['signal'] for r in chunk_records])  
    ages_chunk = np.array([r['age'] for r in chunk_records], dtype=np.float32)
    sexes_chunk = np.array([r['sex'] for r in chunk_records], dtype=np.int8)
    ids_chunk = [r['record_id'] for r in chunk_records]
    dx_chunk = [r['dx'] for r in chunk_records]

    n_chunk = len(chunk_records)
    total_valid += n_chunk

    #  Crear datasets en el primer chunk 
    if hdf5_signal is None:
        dt = h5py.string_dtype(encoding='utf-8')
        hdf5_signal = hdf5_file.create_dataset('signal', shape=(n_chunk, EXPECTED_LEADS, EXPECTED_LENGTH),
                                                   maxshape=(None, EXPECTED_LEADS, EXPECTED_LENGTH),
                                                   dtype='float32', compression='gzip', chunks=(min(64, CHUNK_SIZE), EXPECTED_LEADS, EXPECTED_LENGTH))
        hdf5_age = hdf5_file.create_dataset('age',  shape=(n_chunk,), maxshape=(None,), dtype='float32')
        hdf5_sex = hdf5_file.create_dataset('sex', shape=(n_chunk,), maxshape=(None,), dtype='int8')
        hdf5_record_id = hdf5_file.create_dataset('record_id', shape=(n_chunk,), maxshape=(None,), dtype=dt)
        hdf5_dx = hdf5_file.create_dataset('dx', shape=(n_chunk,), maxshape=(None,), dtype=dt)
    else:
        new_size = write_idx + n_chunk
        hdf5_signal.resize(new_size, axis=0)
        hdf5_age.resize(new_size, axis=0)
        hdf5_sex.resize(new_size, axis=0)
        hdf5_record_id.resize(new_size, axis=0)
        hdf5_dx.resize(new_size, axis=0)

    # Escritura en HDF5
    sl = slice(write_idx, write_idx + n_chunk)
    hdf5_signal[sl] = signals_chunk
    hdf5_age[sl] = ages_chunk
    hdf5_sex[sl] = sexes_chunk
    hdf5_record_id[sl] = ids_chunk
    hdf5_dx[sl] = [json.dumps(d) for d in dx_chunk]

    #  Features ML para Parquet 
    feat = extract_features_chunk(signals_chunk)  # (N_chunk, 60)
    df_chunk = pd.DataFrame(feat, columns=FEATURE_NAMES)
    df_chunk['age'] = ages_chunk
    df_chunk['sex'] = sexes_chunk
    df_chunk['record_id'] = ids_chunk
    df_chunk['dx'] = [json.dumps(d) for d in dx_chunk]
    parquet_chunks.append(df_chunk)

    write_idx += n_chunk

    # Liberar memoria del chunk antes del siguiente
    del signals_chunk, ages_chunk, sexes_chunk, ids_chunk, dx_chunk, chunk_records, feat, df_chunk
    gc.collect()

    log.info('Chunk %d/%d — válidos acumulados: %d',
             chunk_start // CHUNK_SIZE + 1,
             (len(pair_list) + CHUNK_SIZE - 1) // CHUNK_SIZE,
             total_valid)

hdf5_file.close()
print(f'HDF5 cerrado. Registros totales: {total_valid}  |  Fallidos: {total_failed}')

INFO | Chunk 1/91 — válidos acumulados: 500
INFO | Chunk 2/91 — válidos acumulados: 1000
INFO | Chunk 3/91 — válidos acumulados: 1500
INFO | Chunk 4/91 — válidos acumulados: 2000
INFO | Chunk 5/91 — válidos acumulados: 2500
INFO | Chunk 6/91 — válidos acumulados: 3000
INFO | Chunk 7/91 — válidos acumulados: 3500
INFO | Chunk 8/91 — válidos acumulados: 4000
INFO | Chunk 9/91 — válidos acumulados: 4500
INFO | Chunk 10/91 — válidos acumulados: 5000
INFO | Chunk 11/91 — válidos acumulados: 5500
INFO | Chunk 12/91 — válidos acumulados: 6000
INFO | Chunk 13/91 — válidos acumulados: 6500
INFO | Chunk 14/91 — válidos acumulados: 7000
INFO | Chunk 15/91 — válidos acumulados: 7500
INFO | Chunk 16/91 — válidos acumulados: 8000
INFO | Chunk 17/91 — válidos acumulados: 8500
INFO | Chunk 18/91 — válidos acumulados: 9000
INFO | Chunk 19/91 — válidos acumulados: 9500
INFO | Chunk 20/91 — válidos acumulados: 10000
INFO | Chunk 21/91 — válidos acumulados: 10500
INFO | Chunk 22/91 — válidos acumulados: 1

HDF5 cerrado. Registros totales: 45152  |  Fallidos: 0


In [ ]:
df = pd.concat(parquet_chunks, ignore_index=True)
del parquet_chunks
gc.collect()

print("Dataset base:", len(df))

mapped = df["dx"].apply(map_dx)

df["labels"] = mapped.apply(lambda x: x["labels"])
df["categories"] = mapped.apply(lambda x: x["categories"])
df["is_adverse"] = mapped.apply(lambda x: x["is_adverse"])
df["snomed_unknown"] = mapped.apply(lambda x: x["snomed_unknown"])

for cat in CATEGORIES:
    df[f"cat_{cat}"] = df["categories"].apply(lambda x: int(cat in x))

all_dx = df["dx"].apply(json.loads).tolist()

mlb = MultiLabelBinarizer()
y_multi_hot = mlb.fit_transform(all_dx)

dx_classes = mlb.classes_

assert len(df) == len(y_multi_hot), "DESALINEACIÓN ENTRE X E Y"
print("Clases:", len(dx_classes))
print("Shape y:", y_multi_hot.shape)

df["dx_multi_hot"] = list(y_multi_hot)
df.to_parquet(LOCAL_PARQUET, index=False)
print("Dataset final listo:", df.shape)

Dataset base: 45152
Clases: 94
Shape y: (45152, 94)
Clases: 94
Shape y: (45152, 94)
Dataset final listo: (45152, 78)


## 8. Resumen acumulado

In [ ]:
lengths = np.array(all_lengths)
print ('EDA GLOBAL')
print(f'Registros válidos: {total_valid}')
print(f'Fallidos: {total_failed}')
print(f'Longitud señal: min={lengths.min()}  max={lengths.max()}'f'mean={lengths.mean():.1f}  std={lengths.std():.1f}')
print(f'Nulos: {null_stats}')
print(f'Top-10 DX: {dx_counter.most_common(10)}')

EDA GLOBAL
Registros válidos: 45152
Fallidos: 0
Longitud señal: min=5000  max=5000  mean=5000.0  std=0.0
Nulos: {'missing_age': 55, 'missing_sex': 22, 'missing_dx': 0}
Top-10 DX: [('426177001', 16559), ('426783006', 8125), ('164890007', 8060), ('427084000', 7255), ('164934002', 7043), ('55827005', 5401), ('55930002', 4232), ('59931005', 2877), ('427393009', 2550), ('164889003', 1780)]


## 9. Metadata JSON

In [ ]:
metadata = {
    'project':'ECG Big Data Pipeline',
    'version': 'v3-chunked',
    'date':str(datetime.now()),
    'config':{'chunk_size': CHUNK_SIZE, 'max_workers': MAX_WORKERS},
    'stats': {'total_valid': total_valid, 'total_failed': total_failed,
                'null_stats': null_stats},
    'datasets': {
        'deep_learning':    {'format': 'HDF5','path': OUT_HDF5,
                             'shape': [total_valid, EXPECTED_LEADS, EXPECTED_LENGTH]},
        'machine_learning': {'format': 'parquet', 'path': OUT_PARQUET,
                             'shape': list(df.shape)}
    },
    'raw_source': f's3://{BASE_PATH}'
}
with open(LOCAL_META, 'w') as f:
    json.dump(metadata, f, indent=4)
print('Metadata guardada')

Metadata guardada


## 10. Subida a S3

In [ ]:
s3.put(LOCAL_HDF5, OUT_HDF5)
s3.put(LOCAL_PARQUET, OUT_PARQUET)
s3.put(LOCAL_META, OUT_META)
print('Todo subido a S3')

Todo subido a S3
